# How Ask Saurav finds the right facts: tokens, embeddings, chunking and vector stores

This notebook backs up the design decisions in the chatbot with measurements:

1. **Tokenization**: what the embedding model actually sees
2. **Embeddings**: how meaning becomes geometry (cosine similarity)
3. **Chunk size**: why the knowledge base is split into ~700-character chunks
4. **ChromaDB vs FAISS**: same vectors, two vector stores, latency and agreement

Runs on CPU in about a minute (Kaggle, Colab or locally with `pip install -r requirements-dev.txt`).

In [ ]:
import sys, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from app.config import Settings
from app.ingest import load_documents

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL, device="cpu")
questions = [json.loads(l) for l in (ROOT / "eval/questions.jsonl").read_text().splitlines() if l.strip()]
print(f"{len(questions)} evaluation questions, embedding dim = {model.get_sentence_embedding_dimension()}")

## 1. Tokenization

MiniLM uses a **WordPiece** tokenizer with a 30,522-token vocabulary. Common words are single tokens;
rare technical terms are split into sub-word pieces (`##` marks a continuation). The model also has a hard
limit of 256 word pieces per input: anything longer is silently truncated, which is one reason chunks must stay small.

In [ ]:
tok = model.tokenizer
for text in ["Restricted Boltzmann Machine", "R(2+1)D-18 with BiLSTM attention", "ByteTrack", "Saurav built a chatbot"]:
    pieces = tok.tokenize(text)
    print(f"{text!r:42} -> {len(pieces):2d} tokens: {pieces}")
print("max_seq_length =", model.max_seq_length)

In [ ]:
docs = load_documents(Settings())
lengths = pd.Series([len(tok.tokenize(d.page_content)) for d in docs], name="tokens per chunk")
print(lengths.describe().round(1))
print(f"Chunks over the {model.max_seq_length}-token limit: {(lengths > model.max_seq_length).sum()}")
lengths.plot.hist(bins=20, title="Tokens per chunk (current settings)"); plt.axvline(model.max_seq_length, ls="--", c="r"); plt.show()

## 2. Embeddings: meaning as geometry

Each text becomes a 384-dimensional unit vector, so **cosine similarity is just a dot product**.
Paraphrases land close together even with no words in common, which keyword search cannot do.

In [ ]:
sentences = [
    "What is your CGPA?",
    "How good are your grades?",
    "Explain your recommender system research",
    "Tell me about SCRBM",
    "Can you move to Noida?",
    "Are you open to relocation?",
]
E = model.encode(sentences, normalize_embeddings=True)
sim = E @ E.T
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(sentences)), [s[:22] for s in sentences], rotation=40, ha="right")
ax.set_yticks(range(len(sentences)), [s[:22] for s in sentences])
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="w" if sim[i, j] < .6 else "k", fontsize=8)
plt.colorbar(im); plt.title("Cosine similarity: paraphrase pairs light up"); plt.tight_layout(); plt.show()

## 3. Choosing the chunk size

Too small and a chunk loses the context that makes it findable; too large and several topics blur into one
vector (and long chunks get truncated). I rebuild the chunks at several sizes and measure whether the correct
knowledge-base file is retrieved for each evaluation question.

In [ ]:
def retrieval_scores(chunk_size, k=4):
    s = Settings(chunk_size=chunk_size, chunk_overlap=int(chunk_size * 0.17))
    d = load_documents(s)
    D = model.encode([x.page_content for x in d], normalize_embeddings=True, batch_size=32)
    Q = model.encode([q["q"] for q in questions], normalize_embeddings=True)
    top = np.argsort(-(Q @ D.T), axis=1)[:, :k]
    hit1 = hitk = mrr = 0
    for qi, q in enumerate(questions):
        srcs = [d[j].metadata["source"] for j in top[qi]]
        if q["source"] in srcs:
            r = srcs.index(q["source"]) + 1
            hitk += 1; mrr += 1 / r; hit1 += r == 1
    n = len(questions)
    return {"chunk_size": chunk_size, "chunks": len(d), "hit@1": hit1 / n, f"hit@{k}": hitk / n, "mrr": mrr / n}

sweep = pd.DataFrame([retrieval_scores(c) for c in [250, 400, 550, 700, 1000, 1500]]).set_index("chunk_size")
display(sweep.round(3))
sweep[["hit@1", "hit@4", "mrr"]].plot(marker="o", title="Retrieval quality vs chunk size", ylim=(0, 1.05)); plt.show()

## 4. ChromaDB vs FAISS

Both store the same vectors. FAISS is an in-memory similarity-search library (exact `IndexFlatIP` here);
ChromaDB is a full database with persistence, metadata and filtering. For a knowledge base this size both are
exact and fast, so ChromaDB wins on features: the index is built once at Docker build time and loaded from disk.

In [ ]:
import faiss, chromadb, tempfile

d = load_documents(Settings())
D = model.encode([x.page_content for x in d], normalize_embeddings=True).astype("float32")
Q = model.encode([q["q"] for q in questions], normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(D.shape[1]); index.add(D)
t = time.perf_counter(); _, f_ids = index.search(Q, 4); faiss_ms = (time.perf_counter() - t) * 1000 / len(Q)

client = chromadb.PersistentClient(path=tempfile.mkdtemp())
col = client.create_collection("bench", metadata={"hnsw:space": "cosine"})
col.add(ids=[str(i) for i in range(len(d))], embeddings=D.tolist())
t = time.perf_counter(); res = col.query(query_embeddings=Q.tolist(), n_results=4); chroma_ms = (time.perf_counter() - t) * 1000 / len(Q)

agree = np.mean([len(set(map(int, res["ids"][i])) & set(f_ids[i].tolist())) / 4 for i in range(len(Q))])
pd.DataFrame({"ms / query": [faiss_ms, chroma_ms], "persistence": ["no (in memory)", "yes (on disk)"],
              "metadata filters": ["no", "yes"]}, index=["FAISS IndexFlatIP", "ChromaDB (HNSW)"]).assign(
    **{"top-4 agreement with FAISS": [1.0, agree]}).round(3)

## Takeaways

* MiniLM's 256-token limit and the chunk-length histogram show every chunk fits without truncation.
* Paraphrases ("CGPA" and "grades") score far higher than unrelated pairs, which is why semantic retrieval beats keyword search for recruiter questions.
* The chunk-size sweep shows where retrieval quality peaks for this corpus. Production uses 700 characters with ~17% overlap; if the sweep favours a different size, set `CHUNK_SIZE` in `.env` and rerun `python -m eval.retrieval_eval`.
* ChromaDB returns the same neighbours as exact FAISS search, while adding persistence and metadata, so it is the production store.